In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#carregamento do dataset
df = pd.read_csv("../data/processed/safras_mg.csv")

print(f"Safras carregadas: {len(df)}")
print(f"Colunas: {df.columns.tolist()}")
print(df.round(2))

Safras carregadas: 24
Colunas: ['safra', 'chuva_total', 'temp_media', 'temp_min_media', 'et0_total', 'dias_sem_chuva', 'produtividade_sc_ha', 'tendencia']
    safra  chuva_total  temp_media  temp_min_media  et0_total  dias_sem_chuva  \
0    2001        739.7       28.31           19.08     864.35              64   
1    2002       1440.3       26.90           18.73     745.02              48   
2    2003       1223.7       28.06           19.42     813.59              66   
3    2004       1128.3       27.14           18.69     783.55              45   
4    2005       1042.7       27.73           19.14     805.67              50   
5    2006        875.4       27.89           19.23     826.55              59   
6    2007       1130.4       26.73           18.86     718.28              38   
7    2008        926.9       28.28           19.22     869.28              60   
8    2009       1361.3       27.61           19.29     784.77              52   
9    2010        951.1       27.96 

In [2]:
from sklearn.model_selection import train_test_split

#features (entrada)
X = df[["chuva_total", "temp_media", "temp_min_media", "et0_total", "dias_sem_chuva", "tendencia" ]]

#target (previsão)
y = df["produtividade_sc_ha"]

#divisão de treino (80%) e teste (20%)
#random state -> garante que a divisão seja sempre igual
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total de safras: {len(df)}")
print(f"Safras para treino: {len(X_train)}")
print(f"Safras para teste: {len(X_test)}")
print(f"\nFeatures usadas: {X.columns.tolist()}")



Total de safras: 24
Safras para treino: 19
Safras para teste: 5

Features usadas: ['chuva_total', 'temp_media', 'temp_min_media', 'et0_total', 'dias_sem_chuva', 'tendencia']


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

#cria e treina o modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

#faz previsões no conjunto de teste
y_pred = modelo.predict(X_test)

#avalia o modelo
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== RESULTADO DO MODELO ===")
print(f"R² (quando o modelo explica a variação dos dados): {r2:.3f}")
print(f"MAE (erro médio absoluto em sc/ha): {mae:.2f}")

print("\n=== PREVISÕES VS REALIDADE ===")
resultados = pd.DataFrame({
    "safra" : X_test.index.map(lambda i: df.loc[i, "safra"]),
    "real" : y_test.values,
    "previsto" : y_pred.round(10)
})

print(resultados.to_string(index=False))


=== RESULTADO DO MODELO ===
R² (quando o modelo explica a variação dos dados): 0.716
MAE (erro médio absoluto em sc/ha): 5.18

=== PREVISÕES VS REALIDADE ===
 safra  real  previsto
  2009  70.8 72.119039
  2017  90.3 86.063124
  2001  58.2 59.411990
  2019  92.7 90.260559
  2012  59.7 76.414401


In [4]:
#verificação de peso de cada variável

#importância de cada features
importancia = pd.DataFrame({
"feature" : X.columns,
"coeficiente" : modelo.coef_.round(3)
}).sort_values("coeficiente", key=abs, ascending=False)

print("=== PESO DE CADA VARIÁVEL ===")
print(importancia.to_string(index=False))
print(f"\nIntercepto: {modelo.intercept_:.2f}")

=== PESO DE CADA VARIÁVEL ===
       feature  coeficiente
    temp_media      -25.246
temp_min_media       18.457
     tendencia        1.269
dias_sem_chuva       -0.394
     et0_total        0.340
   chuva_total        0.005

Intercepto: 149.19


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# previsão para todas as safras
df["previsto"] = modelo.predict(X).round(1)
df["erro"]     = (df["previsto"] - df["produtividade_sc_ha"]).round(1)

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

#gráfico 1: real vs previsto ao longo do tempo 
ax1 = axes[0]
ax1.plot(df["safra"], df["produtividade_sc_ha"],
         "o-", color="#1D9E75", linewidth=2, markersize=6, label="Real")
ax1.plot(df["safra"], df["previsto"],
         "s--", color="#BA7517", linewidth=2, markersize=6, label="Previsto")

#destaca os pontos de teste
safras_teste = resultados["safra"].values
for safra in safras_teste:
    idx = df[df["safra"] == safra].index[0]
    ax1.axvline(x=safra, color="#E24B4A", alpha=0.2, linewidth=8)

ax1.set_title("AgroPredict — produtividade real vs prevista (sc/ha)\nMilho 1ª safra — Minas Gerais 2001–2024",
              fontsize=13)
ax1.set_xlabel("Safra")
ax1.set_ylabel("Produtividade (sc/ha)")
ax1.legend()
ax1.grid(True, alpha=0.3)

#adiciona faixa de erro
erro_std = df["erro"].std()
ax1.fill_between(df["safra"],
                 df["previsto"] - erro_std,
                 df["previsto"] + erro_std,
                 color="#BA7517", alpha=0.15,
                 label=f"Margem ±{erro_std:.1f} sc/ha")
ax1.legend()

#gráfico 2: erro por safra 
ax2 = axes[1]
cores = ["#E24B4A" if e > 0 else "#1D9E75" for e in df["erro"]]
ax2.bar(df["safra"], df["erro"], color=cores, alpha=0.8)
ax2.axhline(y=0, color="black", linewidth=1)
ax2.axhline(y=mae,  color="#BA7517", linewidth=1.5, linestyle="--",
            label=f"MAE médio: {mae:.1f} sc/ha")
ax2.axhline(y=-mae, color="#BA7517", linewidth=1.5, linestyle="--")
ax2.set_title("Erro do modelo por safra (previsto − real)")
ax2.set_xlabel("Safra")
ax2.set_ylabel("Erro (sc/ha)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/previsao_produtividade.png", dpi=150)
plt.show()
print("Gráfico salvo em outputs/previsao_produtividade.png")

NameError: name 'modelo' is not defined

In [2]:
import pickle
from pathlib import Path

#salva o modelo treinado em arquivo
PASTA_MODELO = Path("../outputs")
PASTA_MODELO.mkdir(exist_ok=True)

with open("../outputs/modelo_agro_predict.pkl", "wb") as f:
    pickle.dump(modelo, f)

print("Modelo salvo em outputs/modelo_agro_predict.pkl")

#salva as métricas finais

metricas = {
    "r2": round(r2, 3),
    "mae": round(mae, 2),
    "margem_erro": round(df["erro"].std(), 2)
}

import json
with open("../outputs/metricas.json", "w") as f:
    json.dump(metricas, f, indent=2)

print("Métricas salvas em outputs/metricas.json")
print(metricas)

NameError: name 'modelo' is not defined